# maniol-elvish ポーズ・画角決定木インスペクション（`pose-inspection-001`）

[`memo.md`](./memo.md) の一次評価レビューで確定した **4 軸決定木アルゴリズム（`facing_direction` → `body_facing / shoulder_tilt_deg` → `top_framing_tag` → `camera_angle_zone`）** を 1 パスで自動実行するノートブックです。

従来の手動フォルダ再コピー＆再計測ループを不要にし、**1 回の推論（`rtmlib` + `WD14-Tagger v3` + `DINOv2`）で全画像を葉ノード（`leaf_id`）へ自動分類し、葉ごとの品質・重複・前後整合チェックまで一気通貫で完了**します。

1. **セットアップ**: 必要パッケージの導入と GPU 確認
2. **Drive マウント & 共通モジュール読み込み**: `colab/snapshot-inspector/common/` のロードと 3 モデル初期化
3. **入力画像準備**: 未分類の画像群を `./data/maniol-elvish/input/` に配置
4. **`maniol-elvish` 決定木・検査ルール設定（v1）**: 4 軸分岐設定と `exclude_looking_back_pose` を含む品質基準
5. **1 パス決定木分類 & 品質検査実行**: 全画像を葉ノードへ自動分類・採否判定
6. **結果レビュー・可視化**: 葉ノード分布・クロス集計・除外理由・任意葉ノードのサムネイル確認
7. **書き出し & Drive ZIP 保存・ランタイム終了メモ**

## 1. セットアップ

In [ ]:
# rtmlib が CPU 版 onnxruntime を引き込んで GPU 推論を潰すのを防ぐため、
# CPU 版を除外してから onnxruntime-gpu を明示的にインストールする
%pip install -q rtmlib opencv-python huggingface_hub transformers pillow pandas matplotlib
%pip uninstall -y onnxruntime
%pip install -q --force-reinstall onnxruntime-gpu

In [ ]:
!nvidia-smi

## 2. Google Drive マウント & 共通モジュール・3 モデル初期化

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/belre/stable-diffusion-pipelines.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/stable-diffusion-pipelines")

# ローカル実行時は親ディレクトリを優先し、Colab 上ではリポジトリを clone/pull する
local_inspector_root = Path("..").resolve()
if (local_inspector_root / "common" / "__init__.py").exists():
    INSPECTOR_ROOT = local_inspector_root
else:
    if not REPO_DIR.exists():
        !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull
    INSPECTOR_ROOT = REPO_DIR / "colab" / "snapshot-inspector"

if str(INSPECTOR_ROOT) not in sys.path:
    sys.path.insert(0, str(INSPECTOR_ROOT))

import os
import copy
import json
import pandas as pd

from common import (
    DEFAULT_DINOV2_REPO,
    DEFAULT_INSPECTION_RULES,
    DEFAULT_TAXONOMY_CONFIG,
    DEFAULT_WD14_REPO,
    DINOv2Embedder,
    WD14Tagger,
    archive_folder_to_drive,
    discover_input_images,
    export_taxonomy_results,
    get_onnx_device_config,
    init_wholebody,
    run_taxonomy_inspection,
    save_results_to_drive,
    show_leaf_summary_chart,
    show_leaf_thumbnails,
)

rtmlib_device, ort_providers = get_onnx_device_config()
print(f"INSPECTOR_ROOT: {INSPECTOR_ROOT}")
print(f"ONNX device: {rtmlib_device} / providers: {ort_providers}")

# 3 モデル初期化（rtmlib Wholebody + WD14-Tagger v3 + DINOv2）
wholebody = init_wholebody(mode="performance", device=rtmlib_device)
wd14_tagger = WD14Tagger(repo_id=DEFAULT_WD14_REPO, providers=ort_providers)
dinov2_embedder = DINOv2Embedder(model_id=DEFAULT_DINOV2_REPO)

## 3. 入力画像準備（`maniol-elvish`）

未分類の画像群をまとめて `./data/maniol-elvish/input/` に配置します（サブフォルダが含まれていても再帰的に読み込みます）。

In [ ]:
CHARACTER_NAME = "maniol-elvish"
INPUT_DIR = f"./data/{CHARACTER_NAME}/input"
OUTPUT_DIR = f"./output/{CHARACTER_NAME}/inspection_001"

!mkdir -p {INPUT_DIR} {OUTPUT_DIR}

# Google Drive からのコピー例（環境に合わせてコメント解除・編集してください）
# !cp '/content/drive/MyDrive/Colab Notebooks/maniol-elvish/raw/'*.png {INPUT_DIR}/
# ZIP から展開する場合:
# !unzip -q '/content/drive/MyDrive/Colab Notebooks/dataset-curation/maniol-elvish/xxx.zip' -d {INPUT_DIR}/

input_items = discover_input_images(INPUT_DIR)
print(f"検出された入力画像数: {len(input_items)} ({INPUT_DIR})")

## 4. `maniol-elvish` 決定木・検査ルール設定（v1 / `memo.md` 準拠）

一次評価レビュー（`memo.md`）で確定した 4 軸決定木と星取表をベースにパラメータを設定します。

- `front`: `body_facing` 分岐なし（全件前向き）、`full_body` は `camera_angle_zone` で分岐
- `three_quarter_left` / `three_quarter_right`: `body_facing` × `top_framing_tag` × `full_body` 仰角分岐
- `side_left` / `side_right`: `body_facing` × `top_framing_tag`（`full_body` は過分割防止のため仰角分岐なし）
- `ambiguous`（`top_framing_score < 0.35`）: 除外せず `camera_angle_zone` 付き救済枠として保持
- `exclude_looking_back_pose_in_front_leaves = True`: `cowboy_shot` / `upper_body` を含む全画角の前向き葉で前後整合チェックを暫定標準適用

In [ ]:
# ==========================================
# 1. 決定木（Taxonomy）分岐設定
# ==========================================
TAXONOMY_CONFIG = copy.deepcopy(DEFAULT_TAXONOMY_CONFIG)
TAXONOMY_CONFIG.update({
    "framing_ambiguous_thr": 0.35,
    "camera_angle_thr": 0.30,
    "body_axis_mode": "body_facing",  # "body_facing" (front/back/twisted) または "shoulder_zone" (front/back_neg/back_pos)
    "split_body_facing_for": [
        "three_quarter_left",
        "three_quarter_right",
        "side_left",
        "side_right",
    ],
    "split_camera_angle_for_full_body": [
        "front",
        "three_quarter_left",
        "three_quarter_right",
    ],
    "split_camera_angle_for_ambiguous": True,
})

# ==========================================
# 2. 葉ノード品質検査ルール（v1）
# ==========================================
INSPECTION_RULES = copy.deepcopy(DEFAULT_INSPECTION_RULES)
INSPECTION_RULES.update({
    # DINOv2 一貫性・葉ノード内重複除外
    "min_visual_centroid_sim": 0.60,
    "min_leaf_centroid_sim": 0.65,
    "min_anchor_sim": None,
    "max_leaf_pair_sim_dedup": 0.96,

    # rtmlib 骨格品質・全画角前後整合
    "require_pose_valid": True,
    "max_person_count": 1,
    "min_body_mean_score": 0.45,
    "proportion_z_max_front": 2.8,
    "proportion_z_max_other": 3.0,
    "exclude_looking_back_pose_in_front_leaves": True,  # cowboy_shot / upper_body 含む全画角で暫定適用

    # WD14 タグ条件
    "required_tags": {
        "1girl": 0.70,
    },
    "forbidden_tags": {
        "monochrome": 0.50,
        "multiple_girls": 0.50,
    },
    "front_facing_forbidden_tags": {
        "from_behind": 0.45,
        "profile": 0.45,
    },
    "front_body_forbidden_tags": {
        "from_behind": 0.45,
    },

    # full_body / eye_level の純度ライン（0.55 で 0.46 俯瞰外れ値を除去 / コア限定なら 0.70〜0.75）
    "full_body_eye_level_min_score": 0.55,
    "full_body_eye_level_max_from_above": 0.35,
})

# 任意: 理想絵柄のアンカー画像パスリスト（未指定なら空リスト）
ANCHOR_PATHS = []

print("Configured TAXONOMY_CONFIG & INSPECTION_RULES for:", CHARACTER_NAME)

## 5. 決定木一括分類 & 品質検査の実行（1 パス完了）

In [ ]:
input_items = discover_input_images(INPUT_DIR)
assert len(input_items) > 0, f"入力画像が見つかりません: {INPUT_DIR}"

result_df, leaf_summary_df, records, general_preds = run_taxonomy_inspection(
    image_items=input_items,
    pose_model=wholebody,
    tagger=wd14_tagger,
    embedder=dinov2_embedder,
    taxonomy_config=TAXONOMY_CONFIG,
    inspection_rules=INSPECTION_RULES,
    anchor_paths=ANCHOR_PATHS or None,
)

n_total = len(result_df)
n_sel = int(result_df["selected"].sum())
print(f"\n=== 決定木一括検査サマリー: 採用 {n_sel} / {n_total} 枚 ({n_sel / n_total * 100:.1f}%) | 葉ノード数: {len(leaf_summary_df)} ===")
display(leaf_summary_df)

## 6. 結果レビュー・可視化

In [ ]:
# 1. 葉ノード別の採用／除外バランスをグラフ表示
show_leaf_summary_chart(leaf_summary_df, top_n=25)

# 2. 主要軸のクロス集計（全体の偏り・密度を確認）
print("=== facing_direction × body_facing ===")
display(pd.crosstab(result_df["facing_direction"], result_df["body_facing"], margins=True))

print("\n=== facing_direction × top_framing_tag ===")
display(pd.crosstab(result_df["facing_direction"], result_df["top_framing_tag"], margins=True))

# 3. 主な除外理由の集計
rej_df = result_df[~result_df["selected"]]
if len(rej_df) > 0:
    print(f"\n=== 除外理由内訳（除外 {len(rej_df)} 枚） ===")
    display(
        rej_df["reject_reasons"]
        .str.split("; ")
        .explode()
        .str.split("(")
        .str[0]
        .value_counts()
        .to_frame("count")
    )
    show_leaf_thumbnails(result_df, selected=False, n=12, cols=4)
else:
    print("\n✅ 除外された画像はありません。")

In [ ]:
# 4. 気になる特定の葉ノードのサムネイルを抜き出して確認（leaf_id を書き換えて実行）
INSPECT_LEAF_ID = leaf_summary_df.index[0]  # デフォルトは最多葉ノード（例: "front/full_body/eye_level"）
print(f"Inspecting leaf: {INSPECT_LEAF_ID}")
show_leaf_thumbnails(result_df, leaf_id=INSPECT_LEAF_ID, selected=True, n=12, cols=4)

## 7. 書き出し & Google Drive ZIP 保存・ランタイム終了メモ

採用画像は `selected/<leaf_id>/` に決定木の階層ごと自動振り分けされ、除外画像は `rejected/<reason>/`（＋`rejected_list.csv`）へ退避されます。

In [ ]:
exported = export_taxonomy_results(
    result_df=result_df,
    leaf_summary_df=leaf_summary_df,
    output_dir=OUTPUT_DIR,
    taxonomy_config=TAXONOMY_CONFIG,
    inspection_rules=INSPECTION_RULES,
    clear_export_dirs=True,
)

In [ ]:
# ==========================================
# Google Drive へのレポート＆ZIP 保存
# ==========================================
DRIVE_OUTPUT_DIR = f"/content/drive/MyDrive/Colab Notebooks/snapshot-inspector/{CHARACTER_NAME}/inspection_001"

save_results_to_drive(
    DRIVE_OUTPUT_DIR,
    file_paths=[
        exported["metrics_csv"],
        exported["summary_csv"],
        exported["config_json"],
    ],
)

# 採用データ（葉ノード別フォルダ構成ごと）を ZIP 化して高速アップロード
archive_folder_to_drive(
    src_dir=exported["selected_dir"],
    drive_output_dir=DRIVE_OUTPUT_DIR,
    zip_name=f"{CHARACTER_NAME}_inspection001_selected.zip",
    local_work_dir=OUTPUT_DIR,
    compress=False,
)

# 除外退避フォルダも Drive へ保存したい場合はコメント解除
# archive_folder_to_drive(
#     src_dir=exported["rejected_dir"],
#     drive_output_dir=DRIVE_OUTPUT_DIR,
#     zip_name=f"{CHARACTER_NAME}_inspection001_rejected.zip",
#     local_work_dir=OUTPUT_DIR,
#     compress=False,
# )

### VS Code 上でのランタイム終了手順
作業が終わったら、GPUランタイムの課金・リソース消費を防ぐため、必ずランタイムを切断・削除してください。

1. VS Code 上部のコマンドパレット（`Ctrl + Shift + P` / macOS は `Cmd + Shift + P`）を開く
2. **`Colab: Remove Server`** を入力して実行する
3. 確認ダイアログが出たら対象の Colab ランタイムを削除して終了する